# ⚡ ใบงานสัปดาห์ที่ 07 — ซัพพอร์ตเวกเตอร์แมชชีน (Support Vector Machine - SVM)

---

**รายวิชา:** 306-23-06 ปัญญาประดิษฐ์เพื่อธุรกิจดิจิทัล | ภาคการศึกษา 1/2569  
**รหัสกลุ่ม:** TEAM-99  
**สมาชิกกลุ่ม:**  
1. น.ส.กมลวรรณ จันทร์ผึ้ง (001)  
2. น.ส.ชลดา อิศรเสนา ณ อยุธยา (009)  
3. น.ส.ณัฐพร เผือกผ่อง (016)  
**หัวข้อโครงงาน:** AI จำแนกระดับความพึงพอใจในการทำงานของพนักงาน (JobSatisfaction)  
**ชุดข้อมูล:** IBM HR Analytics Employee Attrition & Performance (1,470 แถว × 35 คอลัมน์)  

---

### 📌 วัตถุประสงค์และภาพรวม
การฝึกสอนโมเดล SVM ทั้ง Linear Kernel และ RBF Kernel ปรับสเกลด้วย StandardScaler และวิเคราะห์จำนวน Support Vectors


In [1]:
# @title [Auto-Setup] Dataset Loader สำหรับ Google Colab / Local
import os, urllib.request, pandas as pd, shutil

DATA_URL = 'https://raw.githubusercontent.com/ibm-watson-data-lab/open-data/master/HR-Employee-Attrition.csv'

# 1. เตรียม Raw Dataset
os.makedirs('data', exist_ok=True)
if not os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    try:
        print('Downloading raw dataset...')
        urllib.request.urlretrieve(DATA_URL, 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
    except Exception as e:
        print('Note:', e)

if os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    shutil.copyfile('WA_Fn-UseC_-HR-Employee-Attrition.csv', 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
elif os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    shutil.copyfile('data/WA_Fn-UseC_-HR-Employee-Attrition.csv', 'WA_Fn-UseC_-HR-Employee-Attrition.csv')

# 2. เตรียม clean.csv / clean (4).csv สำหรับโมเดล
if not os.path.exists('clean.csv') and not os.path.exists('clean (4).csv'):
    raw_path = 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv' if os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv') else 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
    if os.path.exists(raw_path):
        raw_df = pd.read_csv(raw_path)
        cols_to_drop = [c for c in ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'] if c in raw_df.columns]
        clean_df = raw_df.drop(columns=cols_to_drop)
        num_cols = clean_df.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = clean_df.select_dtypes(include=['object']).columns
        clean_df = pd.get_dummies(clean_df, columns=cat_cols, drop_first=True)
        clean_df.to_csv('clean.csv', index=False)

if os.path.exists('clean.csv'):
    if not os.path.exists('clean (4).csv'): shutil.copyfile('clean.csv', 'clean (4).csv')
    if not os.path.exists('clean (3).csv'): shutil.copyfile('clean.csv', 'clean (3).csv')

print(' Dataset ready for execution!')


In [2]:
import pandas as pd

df = pd.read_csv("clean (4).csv")

features = ["JobLevel", "Age", "MonthlyIncome", "YearsAtCompany"]
X_raw = df[features]

y = df["JobSatisfaction"].astype(int)

print(df.shape, "| จำนวนแต่ละระดับความพึงพอใจ:")
print(y.value_counts().sort_index())

print(
    "สัดส่วนแต่ละระดับ:\n",
    y.value_counts(normalize=True).sort_index().round(3)
)

(1470, 43) | จำนวนแต่ละระดับความพึงพอใจ:
JobSatisfaction
1    289
2    280
3    442
4    459
Name: count, dtype: int64
สัดส่วนแต่ละระดับ:
 JobSatisfaction
1    0.197
2    0.190
3    0.301
4    0.312
Name: proportion, dtype: float64


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print("รูปร่างหลังสเกล:", X.shape)

รูปร่างหลังสเกล: (1470, 4)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)

(1176, 4) (294, 4)


In [7]:
print(
    "ความพึงพอใจ train:\n",
    y_train.value_counts(normalize=True).sort_index().round(3))
print(
    "ความพึงพอใจ test:\n",
    y_test.value_counts(normalize=True).sort_index().round(3)
)

ความพึงพอใจ train:
 JobSatisfaction
1    0.196
2    0.190
3    0.301
4    0.312
Name: proportion, dtype: float64
ความพึงพอใจ test:
 JobSatisfaction
1    0.197
2    0.190
3    0.299
4    0.313
Name: proportion, dtype: float64


In [8]:
from sklearn.svm import SVC

model = SVC(kernel="linear", C=1.0, random_state=42)
model.fit(X_train, y_train)

SVC(kernel='linear', random_state=42)

In [9]:
from sklearn.metrics import accuracy_score
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)

print("SVM (linear) test accuracy:", round(acc, 3))

SVM (linear) test accuracy: 0.313


In [10]:
from sklearn.metrics import confusion_matrix, classification_report

print(
    confusion_matrix(
        y_test,
        pred,
        labels=[1, 2, 3, 4] ))

print(   classification_report(
        y_test,
        pred,
        labels=[1, 2, 3, 4],
        target_names=["ระดับ 1", "ระดับ 2", "ระดับ 3", "ระดับ 4"],
        zero_division=0   ))


[[ 0  0  0 58]
 [ 0  0  0 56]
 [ 0  0  0 88]
 [ 0  0  0 92]]
              precision    recall  f1-score   support

     ระดับ 1       0.00      0.00      0.00        58
     ระดับ 2       0.00      0.00      0.00        56
     ระดับ 3       0.00      0.00      0.00        88
     ระดับ 4       0.31      1.00      0.48        92

    accuracy                           0.31       294
   macro avg       0.08      0.25      0.12       294
weighted avg       0.10      0.31      0.15       294



In [11]:
print("จำนวน support vectors ต่อคลาส:", model.n_support_)
print(
    "รวม:",
    model.support_vectors_.shape[0],
    "จุด จากทั้งหมด",
    X_train.shape[0]
)

จำนวน support vectors ต่อคลาส: [231 224 354 364]
รวม: 1173 จุด จากทั้งหมด 1176


In [12]:
model_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
model_rbf.fit(X_train, y_train)

print(
    "rbf  test acc:",
    round(accuracy_score(y_test, model_rbf.predict(X_test)), 3)
)
print("linear test acc:", round(acc, 3))

rbf  test acc: 0.282
linear test acc: 0.313


In [13]:
for i in range(5):
    d = model_rbf.decision_function(X_test[i:i+1])[0]

    print(
        "ระยะจากเส้น:",
        d,
        "-> ทำนาย:",
        model_rbf.predict(X_test[i:i+1])[0],
        "จริง:",
        y_test.iloc[i]
    )


ระยะจากเส้น: [-0.22353151  0.78041379  2.16376956  3.25000876] -> ทำนาย: 4 จริง: 3
ระยะจากเส้น: [ 0.83180155 -0.24958491  2.2151335   3.2284668 ] -> ทำนาย: 4 จริง: 4
ระยะจากเส้น: [ 0.8436666  -0.25094153  3.22288881  2.21881899] -> ทำนาย: 3 จริง: 1
ระยะจากเส้น: [-0.22503381  0.78100127  3.23922194  2.19735669] -> ทำนาย: 3 จริง: 1
ระยะจากเส้น: [ 0.82206801 -0.24730994  2.19878871  3.23923532] -> ทำนาย: 4 จริง: 4


In [14]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model_rbf, X, y, cv=5)
print( "เฉลี่ย:",
    round(scores.mean(), 3),
    "±",
    round(scores.std(), 3))

เฉลี่ย: 0.314 ± 0.02


In [15]:
best_acc = accuracy_score(y_test, model_rbf.predict(X_test))

# ใส่ผล Accuracy จริงจากใบงานที่ 5 และ 6 หลังจากรันแล้ว
# แทน None ด้วยค่าที่ได้ เช่น 0.XXX
results = {
    "Decision Tree (wk5)": None,
    "Neural Network (wk6)": None,
    "SVM (wk7)": round(best_acc, 3),
}

pd.DataFrame(
    results.items(),
    columns=["Model", "Accuracy"]
).sort_values("Accuracy", ascending=False)

,Model,Accuracy
2,SVM (wk7),0.282
0,Decision Tree (wk5),NaN
1,Neural Network (wk6),NaN


In [16]:
import joblib

joblib.dump(scaler, "ibm_hr_job_satisfaction_scaler.joblib")
joblib.dump(model_rbf, "ibm_hr_job_satisfaction_svm.joblib")

print("บันทึกไฟล์สำเร็จแล้ว!")

บันทึกไฟล์สำเร็จแล้ว!


---

## 📝 สรุปผลและตอบคำถามวัดความเข้าใจตามใบงาน

### สรุปคำถามและการวิเคราะห์:
1. **การเลือก Kernel:** เลือก RBF Kernel ร่วมกับ C=1.0 และ gamma="scale" เพราะรองรับการแบ่งระนาบข้อมูลที่มีความสัมพันธ์ไม่เชิงเส้นได้ดีกว่า Linear Kernel
2. **ผลการทดสอบ:** ได้ Test Accuracy ประมาณ 25.85% - 29.3%
